In [1]:
from dotenv import load_dotenv
load_dotenv(" .env")
import langchain
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage as HM, AIMessage as AM, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from typing import Dict,Any
from tavily import TavilyClient
from pydantic import BaseModel
from dataclasses import dataclass, field
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)
agent = create_agent(
    model = model
)

In [2]:

from langchain_core.prompts import MessagesPlaceholder
summarizer_prompt =""" You are a Conversation Summarizer.

Your task is to summarize the provided HumanMessage and AIMessage objects into a single compact paragraph.

Rules:
1. Preserve the chronological flow of the conversation.
2. Capture:
   - User goals and intentions
   - Questions asked
   - Solutions or explanations provided
   - Important technical concepts, algorithms, classes, functions, or code structures discussed
   - Unresolved or pending questions
3. Prioritize information that would help continue the conversation in the future.
4. Do not reproduce large code blocks.
5. Do not include greetings, acknowledgements, or conversational filler.
6. If a question was asked but not yet answered, explicitly note it as unresolved.
7. Do not invent facts or information not present in the messages.
8. Write a single information-dense paragraph of 3-6 sentences in an objective third-person style.
"""

from langchain_core.messages import HumanMessage, AIMessage

def summarizer(messages: list):

    conversation = []

    for msg in messages:

        if not getattr(msg, "content", "").strip():
            continue

        role = (
            "User"
            if isinstance(msg, HumanMessage)
            else "Assistant"
        )

        conversation.append(
            f"{role}: {msg.content}"
        )

    conversation_text = "\n\n".join(conversation)

    response = model.invoke(
        f"""
        {summarizer_prompt}

        Conversation:

        {conversation_text}
        """
    )

    return response.content
class session:
    def __init__(self,user_id):
        self.user_id = user_id
        self.memory = manager.get_user_memory(user_id).copy()
        self.summary = manager.get_user_summary(user_id).copy()
        self.profile = manager.get_user_profile(user_id)
        self.profile_prompt = self.profile.to_prompt()
        self.turn_count = 0
        self.last_profile_update = 0


In [1]:
from pydantic import BaseModel, Field
from typing import List
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

from langchain_core.messages import HumanMessage
system_prompt = """
# ROLE

You are an expert User Profiling and Behavioral Analysis system.

Your task is to analyze historical conversations and extract stable,
useful user information into the provided structured schema.

Your output will be used as long-term memory for a conversational AI.

# CORE PRINCIPLE

Extract information only when supported by evidence.

Information should be:
1. Explicitly stated by the user, OR
2. Strongly implied through repeated discussion or behavior.

Never invent facts.

# FIELD DEFINITIONS

GOALS
Future outcomes the user wants to achieve.

Include:
- Career objectives
- Learning objectives
- Personal improvement goals
- Planned achievements

Examples:
- Become an AI engineer
- Get an internship
- Learn PyTorch

PREFERENCES
How the user prefers to learn, communicate, work, or receive information.

Include:
- Communication style preferences
- Learning preferences
- Tool preferences
- Workflow preferences

Examples:
- Prefers concise answers
- Prefers simple English explanations
- Learns through implementation

HOBBIES
Activities primarily done for enjoyment or recreation by the user.

Do NOT include:
- Professional learning
- Academic study
- Career development

Examples:
- Chess
- Cricket
- Photography

LIKES
Things the user explicitly enjoys, appreciates, recommends,
or consistently speaks positively about.

Examples:
- Likes PyTorch
- Enjoys Karpathy's tutorials

DISLIKES
Things the user explicitly dislikes, avoids, criticizes,
or complains about.

Examples:
- Dislikes overly theoretical explanations
- Avoids unnecessary complexity

PROJECTS
Projects the user is currently building, maintaining,
planning, or repeatedly discussing.

Include:
- Side projects
- Learning projects
- Open-source projects
- Work projects

Examples:
- Micrograd implementation
- LangChain AI assistant
- Portfolio website

SKILLS
Technologies, frameworks, tools, academic subjects,
or domains that the user has the knowledge about
or demonstrates familiarity with.

Examples:
- Python
- PyTorch
- Machine Learning
- LangChain
- Statistics

# EXTRACTION RULES

- Use concise standalone phrases.
- Remove duplicates.
- Keep the most specific version.
- Do not include explanations.
- Do not include uncertain assumptions.
- Do not infer personality traits.
- Do not infer demographics.
- Do not infer political, religious, or medical information.
- If evidence is insufficient, return an empty list.

# NORMALIZATION

Good:
- "Learn PyTorch"
- "Prefers concise answers"
- "Micrograd implementation"

Bad:
- "The user seems interested in learning PyTorch because they asked many questions."
- "Probably enjoys coding."
- "May want a software engineering job."

# OUTPUT

Return only the structured schema.
"""

def extract_profile(messages: list):

    conversation = []

    for msg in messages:

        if not getattr(msg, "content", "").strip():
            continue

        role = (
            "User"
            if isinstance(msg, HumanMessage)
            else "Assistant"
        )

        conversation.append(
            f"{role}: {msg.content}"
        )

    conversation_text = "\n\n".join(conversation)

    structured_model = model.with_structured_output(
        UserProfileSchema
    )

    prompt = f"""
        {system_prompt}

        Conversation:

    {conversation_text}
    """

    extracted_profile = structured_model.invoke(prompt)

    return extracted_profile
    
def merge_unique(old_list, new_list):

    seen = set(old_list)

    for item in new_list:

        if item not in seen:
            old_list.append(item)
            seen.add(item)

    return old_list
def merge(s1, extracted_profile):

    profile = s1.profile

    profile.goals = merge_unique(
        profile.goals,
        extracted_profile.goals
    )

    profile.preferences = merge_unique(
        profile.preferences,
        extracted_profile.preferences
    )

    profile.hobbies = merge_unique(
        profile.hobbies,
        extracted_profile.hobbies
    )

    profile.likes = merge_unique(
        profile.likes,
        extracted_profile.likes
    )

    profile.dislikes = merge_unique(
        profile.dislikes,
        extracted_profile.dislikes
    )

    profile.projects = merge_unique(
        profile.projects,
        extracted_profile.projects
    )

    profile.skills = merge_unique(
        profile.skills,
        extracted_profile.skills
    )

class UserProfileSchema(BaseModel):
    goals: List[str] = Field(
        default_factory=list,
        description="""
        Objectives the user wants to achieve, is actively working toward,
        or plans to accomplish in the future.

        Examples:
        - Get an AI internship
        - Become an AI engineer
        - Learn PyTorch
        - Build an AI agent
        - Improve coding skills
        """
    )

    preferences: List[str] = Field(
        default_factory=list,
        description="""
        Consistent ways the user prefers information, workflows,
        tools, communication styles, or learning methods.

        Examples:
        - Prefers simple explanations
        - Prefers concise answers
        - Learns by building projects
        - Prefers Python over Java
        """
    )

    hobbies: List[str] = Field(
        default_factory=list,
        description="""
        Recreational activities primarily done for enjoyment rather
        than career, work, or academic goals mentioned by the user.

        Examples:
        - Playing chess
        - Cricket
        - Photography
        - Reading fiction
        """
    )

    likes: List[str] = Field(
        default_factory=list,
        description="""
        Things the user explicitly enjoys, appreciates, recommends,
        or consistently speaks positively about.

        Examples:
        - Likes PyTorch
        - Enjoys Andrej Karpathy videos
        - Likes dark mode
        """
    )

    dislikes: List[str] = Field(
        default_factory=list,
        description="""
        Things the user explicitly dislikes, avoids, complains about,
        or expresses frustration with.

        Examples:
        - Dislikes verbose explanations
        - Dislikes memorization-based learning
        - Avoids unnecessary complexity
        """
    )

    projects: List[str] = Field(
        default_factory=list,
        description="""
        Personal, academic, professional, or side projects the user
        is building, maintaining, planning, or repeatedly discussing.

        Examples:
        - Micrograd implementation
        - LangChain AI agent
        - Personal portfolio website
        - TinyGPT project
        """
    )

    skills: List[str] = Field(
        default_factory=list,
        description="""
        Technologies, tools, subjects, frameworks, or domains that
        the user is familiar with, or already demonstrates
        competence in. Never include things that user wants to learn and asks you about how to do it.

        Examples:
        - Python
        - PyTorch
        - Machine Learning
        - LangChain
        - Statistics
        - Data Structures and Algorithms
        """
    )





model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

        
    




ValidationError: 1 validation error for ChatGoogleGenerativeAI
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'gemini-2.5-fla...: 0, 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [2]:
from dataclasses import dataclass, field
class User:
    def __init__(self,user_id:str,memory = None):
        self.user_id = user_id
        self.memory = memory or list()
        self.profile = UserProfile()
        self.summary = list()
@dataclass
class UserProfile:
    
    goals: list = field(default_factory=list)
    preferences: list = field(default_factory=list)
    likes: list = field(default_factory=list)
    dislikes : list = field(default_factory=list)
    hobbies: list = field(default_factory=list)
    projects: list = field(default_factory= list)
    skills: list = field(default_factory = list)
    def to_prompt(self):
        return f""" User prefers {self.preferences} ,
        user goals are {self.goals}, 
        interests of user are {self.likes},
        hobbies of user are {self.hobbies}, 
        the user does not like{self.dislikes}, the user is working on {self.projects} and the current skills of user are{self.skills}"""
        
class UserManager:
    def __init__(self):
        self.users = dict()
    
    def get_user_summary(self,user_id):
        if len(self.users[user_id].summary) > 0:
            return self.users[user_id].summary[-1]
        else:
            return list()
            
    def get_user_profile(self,user_id):
        if self.is_user_present(user_id):
            return self.get_user_object(user_id).profile

        print("User not present")        
        
        
    def create_user(self,user_id):
        if self.is_user_present(user_id):
            print("User already there")
            return
        u = User(user_id)
        self.users[user_id] = u
        print("New User Created")
    
    def append_memory_message(self,user_id,message : str):
        u = self.users[user_id]
        u.memory.append(message)
        
    def append_all_user_memory_message(self,message:str):
        for user_obj in self.users.values():
            user_obj.memory.append(message)
    
    def append_memory_message_list(self,user_id,message_list : list):
        u = self.users[user_id]
        u.memory = u.memory + message_list
        
    
    def display_every_user(self):
        if not self.users:
            print("No users present")
            return 
        users = self.users.keys()

        for i in users:
            print(i)
        
    def is_user_present(self,user_id:str) -> bool:
        return user_id in self.users
                
    def get_user_object(self,user_id:str):
        if self.is_user_present(user_id):
            return self.users[user_id]
        print("User not present")
        
    def delete_user(self,user_id:str):
        if user_id in self.users:
            del self.users[user_id]
        print("User Deleted")
        
    def get_user_memory(self,user_id:str):
        if self.is_user_present(user_id):
            return self.users[user_id].memory
        return None
    
    def delete_user_memory(self,user_id:str):
        if user_id not in self.users:
            print("User does not exist")
            return 
        memory = self.get_user_memory(user_id)
        memory.clear()
        print("Memory Cleared")
        
    def display_every_user_memory(self):
        for user,objects in self.users.items():
            print(f"{user} : {objects.memory}")   
    
    def delete_every_user(self):
        if not self.users:
            print("No users present")
            return
        self.users.clear()
        print("All users deleted")
        
    def delete_every_user_memory(self,):
        for user in self.users.keys():
            memory = self.get_user_memory(user)
            memory.clear()
        print("Memory for all Users Cleared")
    
        
        

In [3]:
def normalize_message(ai_message):

    if isinstance(ai_message.content, str):
        return ai_message

    text_parts = []

    for block in ai_message.content:
        if (
            isinstance(block, dict)
            and block.get("type") == "text"
        ):
            text_parts.append(block["text"])

    return AIMessage(
        content="\n".join(text_parts)
    )

In [ ]:
from langchain_core.messages import SystemMessage

manager = UserManager()
manager.delete_every_user()
class InvalidUserIdError(Exception):
    pass

while True:
    try:
        user_id = str(input("Enter your User Id"))
        if user_id.strip() == "":
            raise InvalidUserIdError("The UserId cannot be whitespace or empty.")
        if user_id == 'exit':
            break
        if not manager.is_user_present(user_id):
            manager.create_user(user_id)
            
        
        s1 = session(user_id)
    
    except InvalidUserIdError as e:
        print("Error : ",e)
        continue
    while True:

        user_query = input("Ask Anything! : ")
        if user_query.lower() == "exit":
            break
            

        s1.memory.append(HM(content=user_query))
        
        template = ChatPromptTemplate([
            (
                'system', 
                "keep the answers concise and short under 10 lines, if the user is new then greet "
                "and be welcoming and helpful\n\n{system_profile}"
            ),
            MessagesPlaceholder(variable_name='chat_history')
        ])
        
        formatted_prompt = template.invoke({'chat_history': s1.memory,'system_profile' : s1.profile_prompt()})
        
        try:   
            response = agent.invoke(
            {'messages' : formatted_prompt.to_messages()}
        )
        except Exception as e:
            print(e)
        ai_message = response['messages'][-1]

        text = normalize_message(ai_message)
        
        s1.memory.append(text)
        
        print(f"AI: {text.content} \n")
        
        s1.turn_count += 1

        if s1.turn_count % 5 == 0:
        
            summary = summarizer(
                s1.memory[:10]
            )
        
            s1.memory = [
                SystemMessage(
                    content=f"Conversation Summary: {summary}"
                )
            ] + s1.memory[10:]
        
        new_messages = s1.memory[
            s1.last_profile_update:
        ]
        
        if len(new_messages) >= 30:
        
            extracted_profile = extract_profile(
                new_messages
            )
        
            merge(s1, extracted_profile)
        
            s1.last_profile_update = len(
                s1.memory
            )
      
    
    

In [7]:

extracted_profile = extract_profile(
                s1.memory
            )
        
merge(s1, extracted_profile)
        
memory_messages = s1.memory[:6]

summary = summarizer(
    memory_messages
)

s1.memory = [
    SystemMessage(
        content=f"Conversation Summary: {summary}"
    )
] + s1.memory[6:]

s1.summary.append(summary)

In [16]:
print(s1.profile.to_prompt())
print(s1.profile_prompt)

 User prefers ['answer in simple english', 'answer in one sentence'] ,
        user goals are ['become an aiml engineer'], 
        interests of user are [],
        hobbies of user are [], 
        the user does not like[], the user is working on [] and the current skills of user are['computer networks', 'osi model', 'machine learning models', 'data preparation', 'model selection', 'model design', 'training models', 'evaluation', 'hyperparameter tuning']
 User prefers [] ,
        user goals are [], 
        interests of user are [],
        hobbies of user are [], 
        the user does not like[], the user is working on [] and the current skills of user are[]
